# Data Preparation: Test Data
## Version 1

For test data, three domains were explored, along with their corresponding data sources:

- News (from [BBC Burmese International News](https://www.bbc.com/burmese/topics/cnlv9j1z93wt))
- Legal (from [မြန်မာနိုင်ငံကူးလက်မှတ်ဆိုင်ရာဥပဒေ](https://www.moi.gov.mm/file-download/download/public/103555) published by the Myanmar Ministry of Information)
- Religion (from [ခန္ဓဝိပဿနာဘာဝနာ](https://18milecdn.myanmarseo.com/file/18mile-cdn/books/%E1%80%81%E1%80%94%E1%80%B9%E1%80%93%E1%80%9D%E1%80%AD%E1%80%95%E1%80%BF%E1%80%94%E1%80%AC%20%E1%80%98%E1%80%AC%E1%80%9D%E1%80%94%E1%80%AC.pdf) by 18 Miles Sayardaw)

These datasets are then preprocessed and normalized into a standardized format for the language model (LM), primarily for KenLM.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import re

In [2]:
ROOT = Path("..").resolve()

TEST_DIR = ROOT / "data" / "test"
LOG_DIR = TEST_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_TEXT = ROOT / "clean_text.py"

SYL_NORM = ROOT / "syl-normalizer" / "syl_normalizer.py"
SYL_DICT = ROOT / "syl-normalizer" / "final_syl_dictionary_13Feb2024.sorted.txt"

OPPA_WORD = ROOT / "oppaword" / "oppa_word.py"
OPPA_DICT = ROOT / "oppaword" / "myg2p_mypos.dict"

## Load All Datasets

In [3]:
def load_raw(domain: str) -> pd.DataFrame:
    path = TEST_DIR / f"{domain}.raw"
    lines = [
        ln.strip()
        for ln in path.read_text(encoding="utf-8").splitlines()
        if ln.strip()
    ]
    return pd.DataFrame({"text": lines})

In [4]:
df_news = load_raw("news")
df_news.head()

,text
0,ကင်ညာနိုင်ငံမှာ ငွေစက္ကူတွေကို ပန်းစည်းတွေ၊ အလ...


In [5]:
df_legal = load_raw("legal")
df_legal.head()

,text
0,၃၉။ မည်သူမဆို-
1,(က) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(က)ပါ တားမြစ်ချက်တစ်ရပ်ရပ...
2,(ခ) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(ခ)ပါ တားမြစ်ချက်တစ်ရပ်ရပ...
3,(ဂ) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(ဂ)ပါတားမြစ်ချက်တစ်ရပ်ရပ်...
4,(ဃ) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(ဃ) သို့မဟုတ် ပုဒ်မခွဲ(င)...


In [6]:
df_religion = load_raw("religion")
df_religion.head()

,text
0,ဤလောကဓာတ်ကြီးတွင် ငါမရှိ-သူမရှိ ယောင်္ကျား-မိန...
1,(၅)ပါးသဘောသာ အမှန်ရှိကြကုန်၏။ ခန္ဓာ(၅)ပါးဟူသည်...
2,ရူပက္ခန္ဓာ-ဖောက်ပြန်မှုသဘောအစု
3,ဝေဒနက္ခန္ဓာ-ခံစားမှုသဘောအစု
4,သညာက္ခန္ဓာ-မှတ်သားမှုသဘောအစု


## Structured Test Set Construction: Sentence Segmentation

As shown above, directly loading the raw test data does not produce consistent syllable counts per sentence or a consistent maximum syllable token length.

In addition, the dataframes are not displayed properly because the text has not been split into individual sentences. For example, `news.raw` contains only one row even though it includes multiple sentences. In this case, one sentence is not equivalent to one row, which does not meet our intended data structure.

In [7]:
def split_sentences(series):
    # split on Burmese full stop (ပုဒ်မ); keep delimiter on sentence
    out = []
    for text in series.astype(str):
        text = text.strip()
        if not text:
            continue
        parts = re.split(r"(?<=[။])", text)
        for p in parts:
            p = p.strip()
            if p:
                out.append(p)
    return out

In [8]:
df_news_sents = pd.DataFrame({"text": split_sentences(df_news["text"])})
print(len(df_news), "->", len(df_news_sents), "sentences")
df_news_sents.head()

1 -> 17 sentences


,text
0,ကင်ညာနိုင်ငံမှာ ငွေစက္ကူတွေကို ပန်းစည်းတွေ၊ အလ...
1,ကင်ညာနိုင်ငံမှာ ငွေစက္ကူပန်းစည်း လက်ဆောင်ပေးတာ...
2,အထူးသဖြင့် ချစ်သူများနေ့ ဗယ်လင်တိုင်းနေ့ မရော...
3,နာမည်ကြီးသူတွေနဲ့ အွန်လိုင်း ဆယ်လီတွေက အခမ်းအန...
4,ဒီလို ငွေစက္ကူပန်းစည်းတွေ ရအောင် အရောင်အမျိုးမ...


In [9]:
df_legal_sents = pd.DataFrame({"text": split_sentences(df_legal["text"])})
print(len(df_legal), "->", len(df_legal_sents))
df_legal_sents.head()

9 -> 14


,text
0,၃၉။
1,မည်သူမဆို-
2,(က) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(က)ပါ တားမြစ်ချက်တစ်ရပ်ရပ...
3,(ခ) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(ခ)ပါ တားမြစ်ချက်တစ်ရပ်ရပ...
4,(ဂ) ပုဒ်မ ၃၅ ပုဒ်မခွဲ(ဂ)ပါတားမြစ်ချက်တစ်ရပ်ရပ်...


In [10]:
df_religion_sents = pd.DataFrame({"text": split_sentences(df_religion["text"])})
print(len(df_religion), "->", len(df_religion_sents))
df_religion_sents.head()

34 -> 38


,text
0,ဤလောကဓာတ်ကြီးတွင် ငါမရှိ-သူမရှိ ယောင်္ကျား-မိန...
1,(၅)ပါးသဘောသာ အမှန်ရှိကြကုန်၏။
2,ခန္ဓာ(၅)ပါးဟူသည်မှာ…
3,ရူပက္ခန္ဓာ-ဖောက်ပြန်မှုသဘောအစု
4,ဝေဒနက္ခန္ဓာ-ခံစားမှုသဘောအစု


In [11]:
df_news_sents["text"].to_csv(LOG_DIR / "news.raw.tmp", index=False, header=False, lineterminator="\n")
df_legal_sents["text"].to_csv(LOG_DIR / "legal.raw.tmp", index=False, header=False, lineterminator="\n")
df_religion_sents["text"].to_csv(LOG_DIR / "religion.raw.tmp", index=False, header=False, lineterminator="\n")

## Preprocess All Datasets

### Remove Punctuations and Word Tags

The goal of this project is to use KenLM. Therefore, all annotations are removed for now to create a standard corpus.

In [12]:
!python {CLEAN_TEXT} -i {TEST_DIR}/logs/news.raw.tmp -o {TEST_DIR}/news.cleaned.state1

Success! Cleaned text saved to: /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/news.cleaned.state1


In [13]:
!python {CLEAN_TEXT} -i {TEST_DIR}/logs/legal.raw.tmp -o {TEST_DIR}/legal.cleaned.state1

Success! Cleaned text saved to: /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/legal.cleaned.state1


In [14]:
!python {CLEAN_TEXT} -i {TEST_DIR}/logs/religion.raw.tmp -o {TEST_DIR}/religion.cleaned.state1

Success! Cleaned text saved to: /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/religion.cleaned.state1


### Syllable Normalization

In [15]:
!python {SYL_NORM} \
    --dictionary {SYL_DICT} \
    --frequency 2 \
    --input {TEST_DIR}/news.cleaned.state1 \
    --output {TEST_DIR}/news.cleaned.state2 \
    --log {TEST_DIR}/logs/news.cleaned.state2.log \
    --error-output ../data/test/logs/news.cleaned.state2.errors \
    --fuzzy-distance 0

  Fuzzy correction: disabled

=== syl_normalizer summary (v0.6) ===
  Lines processed    :         17
  Tokens processed   :        165
  Passthrough        :          2
  Already valid      :          6

  Fixed - stage 1 (rules, token count)    :        1
  Fixed - stage 1 (individual rule apps)  :        1

  Fixed - stage 2 ngram (dup vowel)       :        0
  Fixed - stage 2 ngram (asat removal)    :        0
  Fixed - stage 2 ngram (char sub)        :        0
  Fixed - stage 2 ngram (medial order)    :        0
  Fixed - stage 2 ngram (dup+asat)        :        0

  Fixed - stage 2 dict (dup vowel)        :        0
  Fixed - stage 2 dict (asat removal)     :        0
  Fixed - stage 2 dict (char sub)         :        0
  Fixed - stage 2 dict (medial order)     :        0
  Fixed - stage 2 dict (dup+asat)         :        0

  Fixed - stage 3 (merge with previous)   :        0
  Fixed - stage 4 (compound split)        :       56

  Total fixed        :         57
  Still unknown

In [16]:
!python {SYL_NORM} \
    --dictionary {SYL_DICT} \
    --frequency 2 \
    --input {TEST_DIR}/legal.cleaned.state1 \
    --output {TEST_DIR}/legal.cleaned.state2 \
    --log {TEST_DIR}/logs/legal.cleaned.state2.log \
    --error-output ../data/test/logs/legal.cleaned.state2.errors \
    --fuzzy-distance 0

  Fuzzy correction: disabled

=== syl_normalizer summary (v0.6) ===
  Lines processed    :         14
  Tokens processed   :        140
  Passthrough        :         15
  Already valid      :          7

  Fixed - stage 1 (rules, token count)    :        0
  Fixed - stage 1 (individual rule apps)  :        0

  Fixed - stage 2 ngram (dup vowel)       :        0
  Fixed - stage 2 ngram (asat removal)    :        0
  Fixed - stage 2 ngram (char sub)        :        0
  Fixed - stage 2 ngram (medial order)    :        0
  Fixed - stage 2 ngram (dup+asat)        :        0

  Fixed - stage 2 dict (dup vowel)        :        0
  Fixed - stage 2 dict (asat removal)     :        0
  Fixed - stage 2 dict (char sub)         :        0
  Fixed - stage 2 dict (medial order)     :        0
  Fixed - stage 2 dict (dup+asat)         :        0

  Fixed - stage 3 (merge with previous)   :        0
  Fixed - stage 4 (compound split)        :       42

  Total fixed        :         42
  Still unknown

In [17]:
!python {SYL_NORM} \
    --dictionary {SYL_DICT} \
    --frequency 2 \
    --input {TEST_DIR}/religion.cleaned.state1 \
    --output {TEST_DIR}/religion.cleaned.state2 \
    --log {TEST_DIR}/logs/religion.cleaned.state2.log \
    --error-output ../data/test/logs/religion.cleaned.state2.errors \
    --fuzzy-distance 0

  Fuzzy correction: disabled

=== syl_normalizer summary (v0.6) ===
  Lines processed    :         38
  Tokens processed   :         92
  Passthrough        :          0
  Already valid      :          1

  Fixed - stage 1 (rules, token count)    :        0
  Fixed - stage 1 (individual rule apps)  :        0

  Fixed - stage 2 ngram (dup vowel)       :        0
  Fixed - stage 2 ngram (asat removal)    :        0
  Fixed - stage 2 ngram (char sub)        :        0
  Fixed - stage 2 ngram (medial order)    :        0
  Fixed - stage 2 ngram (dup+asat)        :        0

  Fixed - stage 2 dict (dup vowel)        :        0
  Fixed - stage 2 dict (asat removal)     :        0
  Fixed - stage 2 dict (char sub)         :        0
  Fixed - stage 2 dict (medial order)     :        0
  Fixed - stage 2 dict (dup+asat)         :        0

  Fixed - stage 3 (merge with previous)   :        0
  Fixed - stage 4 (compound split)        :       34

  Total fixed        :         34
  Still unknown

### Word Segmentation (OppaWord)

In [18]:
!python {OPPA_WORD} \
  --input {TEST_DIR}/news.cleaned.state2 \
  --dict {OPPA_DICT} \
  --output {TEST_DIR}/news.cleaned.state3

In [19]:
!python {OPPA_WORD} \
  --input {TEST_DIR}/legal.cleaned.state2 \
  --dict {OPPA_DICT} \
  --output {TEST_DIR}/legal.cleaned.state3

In [20]:
!python {OPPA_WORD} \
  --input {TEST_DIR}/religion.cleaned.state2 \
  --dict {OPPA_DICT} \
  --output {TEST_DIR}/religion.cleaned.state3

## Checking Datasets

As one can see, the legal test data contains legal section headers in some rows. These rows are removed during further processing.

In [21]:
!head {TEST_DIR}/news.cleaned.state3

ကင် ညာ နိုင် ငံ မှာ ငွေ စက္ကူ တွေ ကို ပန်း စည်း တွေ အ လှ ဆင် ပစ္စည်း တွေ လို လုပ် တာ မျိုး ကို ရပ် တန်း က ရပ် ကြ ဖို့ ဗ ဟို ဘဏ် က သ တိ ပေး လိုက် ပါ တယ်
ကင် ညာ နိုင် ငံ မှာ ငွေ စက္ကူ ပန်း စည်း လက် ဆောင် ပေး တာ က ခေတ် စား လာ နေ ပါ တယ်
အ ထူး သ ဖြ င့် ချစ် သူ များ နေ့ ဗယ် လင် တိုင်း နေ့ မ ရောက် ခင် ငွေ စက္ကူ ပန်း စည်း တွေ မှာ ပို့ လေ့ ရှိ ကြ ပါ တယ်
နာ မည် ကြီး သူ တွေ နဲ့ အွန် လိုင်း ဆယ် လီ တွေ က အ ခမ်း အ နား ပွဲ တွေ မှာ ငွေ စက္ကူ ပန်း စည်း တွေ ကိုင် ထား တဲ့ ဗီ ဒီ ယို တွေ တင် လေ့ ရှိ တာ ကြော င့် လူ တွေ ကြား ရေ ပန်း စား လာ ခဲ့ တာ ဖြစ် ပါ တယ်
ဒီ လို ငွေ စက္ကူ ပန်း စည်း တွေ ရ အောင် အ ရောင် အ မျိုး မျိုး တန် ဖိုး အ မျိုး မျိုး ငွေ စက္ကူ တွေ ကို လိပ် ပြီး ပန်း စည်း ကြီး တစ် ခု လို ပုံ စံ ဖြစ် အောင် လုပ် လေ့ ရှိ ပါ တယ်
ဒါ ပေ မဲ့ ဒီ လို လုပ် တာ ဟာ ကင် ညာ နိုင် ငံ သုံး ငွေ ကို သိက္ခာ ကျ ဆင်း တန် ဖိုး မဲ့ စေ ပြီး မိ ရင် ထောင် ဒဏ် ၇ နှစ် ကျ ခံ နိုင် တယ် လို့ ကင် ညာ ဗ ဟို ဘဏ် က ပြော ပါ တယ်
ဒီ ငွေ စက္ကူ တွေ ကို ပန်း စည်း ကြီး ဖြစ် အောင် ခေါက် ချိုး လိပ် ကော် နဲ့ ကပ် က လစ် နဲ့ တွယ် ပြီး တစ် ခု နဲ့ တစ် ခ

In [22]:
!head {TEST_DIR}/legal.cleaned.state3

၃၉
မည် သူ မ ဆို
က ပုဒ် မ ၃၅ ပုဒ် မ ခွဲ က ပါ တား မြစ် ချက် တစ် ရပ် ရပ် ကို ဖောက် ဖျက် ကျူး လွန် ကြောင်း ပြစ်မှု ထင် ရှား စီ ရင် ခြင်း ခံ ရ လျှင် ထို သူ ကို အ နည်း ဆုံး ထောင် ဒဏ် သုံး လ မှ အ များ ဆုံး ထောင် ဒဏ် တစ် နှစ် အ ထိ ဖြစ် စေ ငွေ ဒဏ် ငါး သိန်း ကျပ် ဖြစ် စေ ဒဏ် နှစ် ရပ် လုံး ဖြစ် စေ ချ မှတ် ရ မည်
ခ ပုဒ် မ ၃၅ ပုဒ် မ ခွဲ ခ ပါ တား မြစ် ချက် တစ် ရပ် ရပ် ကို ဖောက် ဖျက် ကျူး လွန် ကြောင်း ပြစ်မှု ထင် ရှား စီ ရင် ခြင်း ခံ ရ လျှင် ထို သူ ကို အ နည်း ဆုံး ထောင် ဒဏ် သုံး လ မှ အ များ ဆုံး ထောင် ဒဏ် တစ် နှစ် ချ မှတ် ရ မည့် အ ပြင် အ နည်း ဆုံး ငွေ ဒဏ် သုံး သိန်း ကျပ် မှ အ များ ဆုံး ငွေ ဒဏ် ငါး သိန်း ကျပ် အ ထိ ချ မှတ် နိုင် မည်
ဂ ပုဒ် မ ၃၅ ပုဒ် မ ခွဲ ဂ ပါ တား မြစ် ချက် တစ် ရပ် ရပ် ကို ဖောက် ဖျက် ကျူး လွန် ကြောင်း ပြစ် မှု ထင် ရှား စီ ရင် ခြင်း ခံ ရ လျှင် ထို သူ ကို အ နည်း ဆုံး ထောင် ဒဏ် ခြောက် လ မှ အ များ ဆုံး ထောင် ဒဏ် ငါး နှစ် အ ထိ ချ မှတ် ရ မည့် အ ပြင် အ နည်း ဆုံး ငွေ ဒဏ် ငါး သိန်း ကျပ် မှ အ များ ဆုံး ငွေ ဒဏ် ၁၀ သိန်း ကျပ် အ ထိ ချ မှတ် နိုင် မည်
ဃ ပုဒ် မ ၃၅ ပုဒ် မ ခွဲ ဃ သို့ မ ဟုတ် ပုဒ် မ ခွဲ င 

In [23]:
!head {TEST_DIR}/religion.cleaned.state3

ဤ လော က ဓာတ် ကြီး တွင် ငါ မ ရှိ သူ မ ရှိ ယောင်္ကျား မိန်း မ ဟူ၍ မ ရှိ ခန္ဓာ
၅ ပါး သ ဘော သာ အ မှန် ရှိ ကြ ကုန်၏
ခန္ဓာ၅ ပါး ဟူ သည် မှာ…
ရူပက္ခန္ဓာ ဖောက် ပြန် မှု သ ဘော အ စု
ဝေဒနက္ခန္ဓာ ခံ စား မှု သ ဘော အ စု
သ ညာက္ခန္ဓာ မှတ် သား မှု သ ဘော အ စု
သင်္ခါရက္ခန္ဓာ ဝေ ဖန် မှု သ ဘော အ စု
ဝိညာဏက္ခန္ဓာ သိ မှု သ ဘော အ စု တို့ ဖြစ် ကြ ပါ သည်
ရူပက္ခန္ဓာ ဟု ဆို အပ် သော ဟော ဒီ အတ္တ ဘော ကြီး ကို ၄ ပါး သော
ဓာတ် တို့ ဖြင့် သာ ဓာတ် တို့ ဖြင့် သာ ဖွဲ့ စည်း ထား အပ် ကုန်၏


## Structured Test Set Construction: Data Standardization

The test datasets must be standardized across all domains to ensure a fair comparison of the LM performance. Each dataset is truncated to exactly 300 syllable tokens, with each sentence containing 20 syllable tokens, resulting in a total of 15 sentences per dataset.

In addition, sentences containing fewer than 5 syllable tokens are removed to prevent legal section headers from being counted as valid sentences. This approach is used because directly removing numbers would also unintentionally remove meaningful numeric content appearing within normal sentences.

In [24]:
MIN_SYLS_PER_SENT = 2  # a sentence must contain at least this many syllables
SYLS_PER_SENT = 20  # this many syllables per sentence
SENTS_PER_DOC = 15  # this many sentences per document
SYLS_PER_DOC = SYLS_PER_SENT * SENTS_PER_DOC  # this many syllables per document for each domain

In [25]:
df_news_state3 = pd.read_csv(f"{TEST_DIR}/news.cleaned.state3", header=None, names=["text"])
df_legal_state3 = pd.read_csv(f"{TEST_DIR}/legal.cleaned.state3", header=None, names=["text"])
df_religion_state3 = pd.read_csv(f"{TEST_DIR}/religion.cleaned.state3", header=None, names=["text"])

dfs = {
    "news": df_news_state3,
    "legal": df_legal_state3,
    "religion": df_religion_state3,
}

kept = {}
dropped_log = []

In [26]:
# remove empty lines and legal headers
for name, df in dfs.items():
    df = df.copy()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].ne("")].reset_index(drop=True)
    df["n_syl"] = df["text"].str.split().str.len()

    drop = df[df["n_syl"] < MIN_SYLS_PER_SENT].copy()
    keep = df[df["n_syl"] >= MIN_SYLS_PER_SENT].copy()

    kept[name] = keep[["text"]].reset_index(drop=True)
    for _, row in drop.iterrows():
        dropped_log.append({
            "domain": name,
            "n_syl": row["n_syl"],
            "text": row["text"][:120],
        })

    print(
        f"{name}: total={len(df)} kept={len(keep)} dropped={len(drop)} "
        f"syllables_kept={keep['n_syl'].sum()}"
    )

df_dropped = pd.DataFrame(dropped_log)
df_dropped

news: total=17 kept=17 dropped=0 syllables_kept=659
legal: total=14 kept=9 dropped=5 syllables_kept=557
religion: total=38 kept=38 dropped=0 syllables_kept=404


,domain,n_syl,text
0,legal,1,၃၉
1,legal,1,၄၀
2,legal,1,၄၁
3,legal,1,၄၂
4,legal,1,၄၃


In [27]:
# get syllable tokens for each domain
pools = {}
for name, df in kept.items():
    tokens = []
    for line in df["text"]:
        tokens.extend(line.split())
    pools[name] = tokens
    print(
        f"{name}:",
        f"sentences={len(df)}",
        f"syllables={len(tokens)}",
        f"possible_docs={len(tokens) // SYLS_PER_DOC}",
    )

news: sentences=17 syllables=659 possible_docs=2
legal: sentences=9 syllables=557 possible_docs=1
religion: sentences=38 syllables=404 possible_docs=1


In [28]:
# get the minimum number of documents that can be created from the syllable tokens
n_docs = min(len(tokens) // SYLS_PER_DOC for tokens in pools.values())
if n_docs < 1:
    raise ValueError(f"Not enough syllables for one {SYLS_PER_DOC}-syl document in all domains.")

print("aligned n_docs:", n_docs)
print("lines per .clean file:", n_docs * SENTS_PER_DOC)

grid_lines = {}

# create standardized documents for each domain
for name, tokens in pools.items():
    usable = n_docs * SYLS_PER_DOC
    tokens = tokens[:usable]

    lines = []
    for i in range(0, len(tokens), SYLS_PER_SENT):
        chunk = tokens[i : i + SYLS_PER_SENT]
        if len(chunk) == SYLS_PER_SENT:
            lines.append(" ".join(chunk))

    lines = lines[: n_docs * SENTS_PER_DOC]
    assert len(lines) == n_docs * SENTS_PER_DOC
    assert all(len(ln.split()) == SYLS_PER_SENT for ln in lines)

    grid_lines[name] = lines
    out = TEST_DIR / f"{name}.cleaned.state4"
    out.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"{name} -> {out}")

aligned n_docs: 1
lines per .clean file: 15
news -> /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/news.cleaned.state4
legal -> /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/legal.cleaned.state4
religion -> /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/religion.cleaned.state4


In [29]:
# verify the number of lines and syllables in the standardized documents
for name in ("news", "legal", "religion"):
    lines = [
        ln.strip()
        for ln in (TEST_DIR / f"{name}.cleaned.state4").read_text(encoding="utf-8").splitlines()
        if ln.strip()
    ]
    print(
        f"{name}:",
        f"lines={len(lines)}",
        f"expected={n_docs * SENTS_PER_DOC}",
        f"words={sum(len(ln.split()) for ln in lines)}",
    )

news: lines=15 expected=15 words=300
legal: lines=15 expected=15 words=300
religion: lines=15 expected=15 words=300


## Load Final Datasets

In [30]:
final_df_news = pd.read_csv(f"{TEST_DIR}/news.cleaned.state4", header=None, names=["text"])
final_df_news

,text
0,ကင် ညာ နိုင် ငံ မှာ ငွေ စက္ကူ တွေ ကို ပန်း စည်...
1,မျိုး ကို ရပ် တန်း က ရပ် ကြ ဖို့ ဗ ဟို ဘဏ် က သ...
2,နိုင် ငံ မှာ ငွေ စက္ကူ ပန်း စည်း လက် ဆောင် ပေး...
3,သ ဖြ င့် ချစ် သူ များ နေ့ ဗယ် လင် တိုင်း နေ့ မ...
4,ပို့ လေ့ ရှိ ကြ ပါ တယ် နာ မည် ကြီး သူ တွေ နဲ့ ...
5,အ နား ပွဲ တွေ မှာ ငွေ စက္ကူ ပန်း စည်း တွေ ကိုင...
6,တာ ကြော င့် လူ တွေ ကြား ရေ ပန်း စား လာ ခဲ့ တာ ...
7,စည်း တွေ ရ အောင် အ ရောင် အ မျိုး မျိုး တန် ဖို...
8,ပန်း စည်း ကြီး တစ် ခု လို ပုံ စံ ဖြစ် အောင် လု...
9,လုပ် တာ ဟာ ကင် ညာ နိုင် ငံ သုံး ငွေ ကို သိက္ခာ...


In [31]:
final_df_legal = pd.read_csv(f"{TEST_DIR}/legal.cleaned.state4", header=None, names=["text"])
final_df_legal

,text
0,မည် သူ မ ဆို က ပုဒ် မ ၃၅ ပုဒ် မ ခွဲ က ပါ တား မ...
1,ဖောက် ဖျက် ကျူး လွန် ကြောင်း ပြစ်မှု ထင် ရှား ...
2,ထောင် ဒဏ် သုံး လ မှ အ များ ဆုံး ထောင် ဒဏ် တစ် ...
3,ကျပ် ဖြစ် စေ ဒဏ် နှစ် ရပ် လုံး ဖြစ် စေ ချ မှတ်...
4,ခ ပါ တား မြစ် ချက် တစ် ရပ် ရပ် ကို ဖောက် ဖျက် ...
5,ခံ ရ လျှင် ထို သူ ကို အ နည်း ဆုံး ထောင် ဒဏ် သု...
6,နှစ် ချ မှတ် ရ မည့် အ ပြင် အ နည်း ဆုံး ငွေ ဒဏ်...
7,ဒဏ် ငါး သိန်း ကျပ် အ ထိ ချ မှတ် နိုင် မည် ဂ ပု...
8,မြစ် ချက် တစ် ရပ် ရပ် ကို ဖောက် ဖျက် ကျူး လွန်...
9,လျှင် ထို သူ ကို အ နည်း ဆုံး ထောင် ဒဏ် ခြောက် ...


In [32]:
final_df_religion = pd.read_csv(f"{TEST_DIR}/religion.cleaned.state4", header=None, names=["text"])
final_df_religion

,text
0,ဤ လော က ဓာတ် ကြီး တွင် ငါ မ ရှိ သူ မ ရှိ ယောင်...
1,ပါး သ ဘော သာ အ မှန် ရှိ ကြ ကုန်၏ ခန္ဓာ၅ ပါး ဟူ...
2,အ စု ဝေဒနက္ခန္ဓာ ခံ စား မှု သ ဘော အ စု သ ညာက္ခ...
3,ဝေ ဖန် မှု သ ဘော အ စု ဝိညာဏက္ခန္ဓာ သိ မှု သ ဘေ...
4,ဟု ဆို အပ် သော ဟော ဒီ အတ္တ ဘော ကြီး ကို ၄ ပါး ...
5,သာ ဖွဲ့ စည်း ထား အပ် ကုန်၏ ထို တွင်… ပ ထ ဝီ သည...
6,လေး တဲ့ သ ဘော ပေါ့ တဲ့ သ ဘော ကြမ်း တဲ့ သ ဘော ခ...
7,ကျင် ကိုက် ခဲ ထုံ ကျင် တတ် သော သ ဘော ရှိ ကြ ကု...
8,သော သ ဘော ရှိ၏ တေ ဇော သည် ပူ တတ် အေး တတ် ရင့် ...
9,ယော သည် တွန်း ကန် လှုပ် ရှား ရွှေ့လျား ထောက် က...


## Exploring the Final Dataset File Size

In [33]:
!ls -lh {TEST_DIR}/news.raw {TEST_DIR}/news.cleaned.state4

-rw-rw-r-- 1 lawun330 lawun330 3.1K May 16 00:47 /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/news.cleaned.state4
-rw-rw-r-- 1 lawun330 lawun330 6.1K May 15 17:22 /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/news.raw


In [34]:
!ls -lh {TEST_DIR}/legal.raw {TEST_DIR}/legal.cleaned.state4

-rw-rw-r-- 1 lawun330 lawun330 3.1K May 16 00:47 /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/legal.cleaned.state4
-rw-rw-r-- 1 lawun330 lawun330 5.4K May 15 18:18 /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/legal.raw


In [35]:
!ls -lh {TEST_DIR}/religion.raw {TEST_DIR}/religion.cleaned.state4

-rw-rw-r-- 1 lawun330 lawun330 3.1K May 16 00:47 /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/religion.cleaned.state4
-rw-rw-r-- 1 lawun330 lawun330 3.8K May 15 22:17 /home/lawun330/Desktop/burmese-domain-specific-lm/data/test/religion.raw


## References
- ```bibtex
  @misc{syl_normalizer,
    author       = {Ye Kyaw Thu},
    title        = {{Syllable Normalization Tool for Myanmar Language}},
    version      = {0.6},
    month        = {May},
    year         = {2026},
    publisher    = {GitHub},
    url          = {https://github.com/ye-kyaw-thu/syl-Normalizer/tree/main/ver_0.6},
    note         = {Accessed: YYYY-MM-DD},
    institution  = {Language Understanding Lab (LU Lab), Myanmar}
  }
  ```

- ```bibtex
  @misc{oppaWord_2025,
    author       = {Ye Kyaw Thu},
    title        = {{oppaWord: Hybrid DAG+Bi-MM+LM Myanmar Word Segmenter}},
    version      = {1.0},
    month        = {August},
    year         = {2025},
    publisher    = {GitHub},
    url          = {https://github.com/ye-kyaw-thu/oppaWord},
    note         = {Accessed: YYYY-MM-DD},
    institution  = {Language Understanding Lab (LU Lab), Myanmar}
  }
  ```